# CapitalFlow AI — Data Generation & Model Training
This notebook builds the synthetic New Capital traffic dataset and trains the models used by `app.py`.

The road network covers:
- **Gateway roads**: Cairo-Suez Road, Cairo-Ain Sokhna Road, Middle Ring Road, Regional Ring Road, Mohamed Bin Zayed Axis (North/South), Al-Amal Axis
- **Internal roads**: Central Business District (CBD), Government District (Ministries Area), Financial District (Bank Area), C3 District, C7 District, Diplomatic Quarter, Green River Corridor

## 1. Import Libraries

In [4]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("Libraries imported successfully!")

Libraries imported successfully!


In [5]:
import sklearn, numpy, joblib
print(sklearn.__version__, numpy.__version__, joblib.__version__)

1.6.1 2.1.3 1.4.2


## 2. Road Network Definition
No access to real New Capital traffic sensor data was available, so this section defines a synthetic (simulated) dataset generator that mimics realistic city traffic behavior: rush-hour peaks, weekend dips, weather effects, and holiday effects.

In [6]:
np.random.seed(42)

GATEWAY_ROADS = [
    "Cairo-Suez Road",
    "Cairo-Ain Sokhna Road",
    "Middle Ring Road",
    "Regional Ring Road",
    "Mohamed Bin Zayed Axis (North)",
    "Mohamed Bin Zayed Axis (South)",
    "Al-Amal Axis",
]

INTERNAL_ROADS = [
    "Central Business District (CBD)",
    "Government District (Ministries Area)",
    "Financial District (Bank Area)",
    "C3 District",
    "C7 District",
    "Diplomatic Quarter",
    "Green River Corridor",
]

ALL_ROADS = GATEWAY_ROADS + INTERNAL_ROADS

ROAD_TYPE = {r: "Gateway" for r in GATEWAY_ROADS}
ROAD_TYPE.update({r: "Internal" for r in INTERNAL_ROADS})

# Baseline hourly capacity per road (vehicles/hour)
BASE_VOLUME = {
    "Cairo-Suez Road": 2400,
    "Cairo-Ain Sokhna Road": 1600,
    "Middle Ring Road": 2100,
    "Regional Ring Road": 2600,
    "Mohamed Bin Zayed Axis (North)": 1900,
    "Mohamed Bin Zayed Axis (South)": 1800,
    "Al-Amal Axis": 1500,
    "Central Business District (CBD)": 1300,
    "Government District (Ministries Area)": 1100,
    "Financial District (Bank Area)": 1250,
    "C3 District": 1000,
    "C7 District": 950,
    "Diplomatic Quarter": 800,
    "Green River Corridor": 700,
}

print("Road network defined:", len(ALL_ROADS), "roads")

Road network defined: 14 roads


## 3. Holidays, Weather States, and Hourly Traffic Profile

In [7]:
HOLIDAYS_BY_DATE = {
    "2022-01-07": "Coptic Christmas",
    "2022-01-25": "Revolution Day (Jan 25)",
    "2022-04-25": "Sinai Liberation Day",
    "2022-05-01": "Labour Day",
    "2022-05-02": "Eid al-Fitr",
    "2022-05-03": "Eid al-Fitr",
    "2022-06-30": "June 30 Revolution",
    "2022-07-09": "Eid al-Adha",
    "2022-07-10": "Eid al-Adha",
    "2022-07-23": "Revolution Day (Jul 23)",
    "2022-10-06": "Armed Forces Day",
    "2023-01-07": "Coptic Christmas",
    "2023-01-25": "Revolution Day (Jan 25)",
    "2023-04-21": "Eid al-Fitr",
    "2023-04-22": "Eid al-Fitr",
    "2023-04-25": "Sinai Liberation Day",
    "2023-05-01": "Labour Day",
    "2023-06-28": "Eid al-Adha",
    "2023-06-29": "Eid al-Adha",
    "2023-06-30": "June 30 Revolution",
    "2023-07-23": "Revolution Day (Jul 23)",
}

WEATHER_STATES = [
    ("Clear", "clear sky", 0.55),
    ("Clouds", "scattered clouds", 0.20),
    ("Clouds", "few clouds", 0.08),
    ("Clouds", "broken clouds", 0.05),
    ("Rain", "light rain", 0.05),
    ("Rain", "moderate rain", 0.02),
    ("Haze", "haze", 0.03),
    ("Dust", "sand/dust whirls", 0.02),
]

weather_pool = [w[:2] for w in WEATHER_STATES]
weather_weights = [w[2] for w in WEATHER_STATES]


def sample_weather(rng):
    idx = rng.choice(len(weather_pool), p=weather_weights)
    return weather_pool[idx]


def hourly_factor(hour, is_weekend, road_type):
    """Return a multiplier representing how busy an hour is."""

    if is_weekend:
        profile = {
            0: 0.35, 1: 0.25, 2: 0.20, 3: 0.18, 4: 0.20, 5: 0.30,
            6: 0.45, 7: 0.60, 8: 0.70, 9: 0.80, 10: 0.90, 11: 0.95,
            12: 1.00, 13: 1.00, 14: 0.95, 15: 0.90, 16: 0.90,
            17: 0.95, 18: 1.05, 19: 1.10, 20: 1.05, 21: 0.90,
            22: 0.70, 23: 0.50,
        }
    else:
        if road_type == "Gateway":
            profile = {
                0: 0.15, 1: 0.10, 2: 0.08, 3: 0.08, 4: 0.12, 5: 0.30,
                6: 0.65, 7: 1.00, 8: 1.15, 9: 0.85, 10: 0.55, 11: 0.55,
                12: 0.60, 13: 0.65, 14: 0.65, 15: 0.70, 16: 0.90,
                17: 1.10, 18: 1.20, 19: 1.05, 20: 0.75, 21: 0.55,
                22: 0.35, 23: 0.22,
            }
        else:
            profile = {
                0: 0.10, 1: 0.08, 2: 0.06, 3: 0.06, 4: 0.08, 5: 0.15,
                6: 0.35, 7: 0.65, 8: 0.95, 9: 1.10, 10: 1.00, 11: 0.95,
                12: 0.90, 13: 0.95, 14: 0.90, 15: 0.85, 16: 0.80,
                17: 0.85, 18: 0.75, 19: 0.55, 20: 0.40, 21: 0.30,
                22: 0.20, 23: 0.14,
            }

    return profile[hour]


print("Holiday calendar, weather states, and hourly profiles ready.")

Holiday calendar, weather states, and hourly profiles ready.


## 4. Generate the Synthetic Dataset

In [8]:
def generate_dataset(start="2022-01-01", end="2023-03-31", freq="h", seed=42):

    rng = np.random.default_rng(seed)
    timestamps = pd.date_range(start=start, end=end, freq=freq)

    rows = []

    for road in ALL_ROADS:

        road_type = ROAD_TYPE[road]
        base = BASE_VOLUME[road]

        for ts in timestamps:

            date_str = ts.strftime("%Y-%m-%d")
            holiday = HOLIDAYS_BY_DATE.get(date_str, None)

            # Egypt's weekend is Friday-Saturday (dayofweek: Mon=0 ... Sun=6)
            is_weekend = ts.dayofweek in (4, 5)

            hf = hourly_factor(ts.hour, is_weekend, road_type)

            weather_main, weather_description = sample_weather(rng)

            day_of_year = ts.dayofyear
            seasonal = 22 + 10 * np.sin(2 * np.pi * (day_of_year - 100) / 365)
            daily_swing = 5 * np.sin(2 * np.pi * (ts.hour - 15) / 24)
            temp = seasonal + daily_swing + rng.normal(0, 1.5)

            if weather_main == "Rain":
                rain_1h = round(float(rng.uniform(0.2, 6.0)), 1)
            else:
                rain_1h = 0.0

            snow_1h = 0.0

            if weather_main == "Clear":
                clouds_all = int(rng.integers(0, 15))
            elif weather_main == "Clouds":
                clouds_all = int(rng.integers(20, 80))
            elif weather_main == "Rain":
                clouds_all = int(rng.integers(60, 100))
            else:
                clouds_all = int(rng.integers(10, 50))

            volume = base * hf

            if holiday is not None:
                if road_type == "Gateway":
                    volume *= 1.35
                else:
                    volume *= 0.45

            if weather_main == "Rain":
                volume *= 0.85
            elif weather_main == "Dust":
                volume *= 0.80
            elif weather_main == "Haze":
                volume *= 0.92

            volume *= rng.normal(1.0, 0.08)
            volume = max(50, volume)

            rows.append({
                "holiday": holiday,
                "temp": round(float(temp), 2),
                "rain_1h": rain_1h,
                "snow_1h": snow_1h,
                "clouds_all": clouds_all,
                "weather_main": weather_main,
                "weather_description": weather_description,
                "date_time": ts.strftime("%m/%d/%Y %H:%M"),
                "road_name": road,
                "road_type": road_type,
                "traffic_volume": int(round(volume)),
            })

    return pd.DataFrame(rows)


df = generate_dataset()

print("Generated shape:", df.shape)
df.head(10)

Generated shape: (152558, 11)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,road_name,road_type,traffic_volume
0,NaN,14.06,0.0,0.0,45,Clouds,few clouds,01/01/2022 00:00,Cairo-Suez Road,Gateway,903
1,NaN,12.64,0.0,0.0,12,Clear,clear sky,01/01/2022 01:00,Cairo-Suez Road,Gateway,606
2,NaN,13.36,0.0,0.0,70,Clouds,few clouds,01/01/2022 02:00,Cairo-Suez Road,Gateway,514
3,NaN,12.19,5.0,0.0,78,Rain,light rain,01/01/2022 03:00,Cairo-Suez Road,Gateway,381
4,NaN,11.35,0.0,0.0,13,Clear,clear sky,01/01/2022 04:00,Cairo-Suez Road,Gateway,514
5,NaN,9.31,0.0,0.0,23,Clouds,scattered clouds,01/01/2022 05:00,Cairo-Suez Road,Gateway,681
6,NaN,8.32,0.0,0.0,37,Haze,haze,01/01/2022 06:00,Cairo-Suez Road,Gateway,966
7,NaN,8.31,0.0,0.0,11,Clear,clear sky,01/01/2022 07:00,Cairo-Suez Road,Gateway,1488
8,NaN,10.47,0.0,0.0,41,Clouds,scattered clouds,01/01/2022 08:00,Cairo-Suez Road,Gateway,1611
9,NaN,8.01,0.0,0.0,14,Clear,clear sky,01/01/2022 09:00,Cairo-Suez Road,Gateway,2093


In [9]:
df.to_csv("New_Capital_Roads_Traffic_Volume.csv", index=False)
print("Saved New_Capital_Roads_Traffic_Volume.csv")

Saved New_Capital_Roads_Traffic_Volume.csv


## 5. Feature Engineering

In [10]:
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"]).copy()
df = df.sort_values("date_time").reset_index(drop=True)

df["year"] = df["date_time"].dt.year
df["month"] = df["date_time"].dt.month
df["day"] = df["date_time"].dt.day
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek

df["is_weekend"] = (df["day_of_week"].isin([4, 5])).astype(int)

df["is_rush_hour"] = (
    ((df["hour"] >= 7) & (df["hour"] <= 9))
    | ((df["hour"] >= 16) & (df["hour"] <= 19))
).astype(int)

print("Dataset ready:", df.shape)
df.head()

Dataset ready: (152558, 18)


,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,road_name,road_type,traffic_volume,year,month,day,hour,day_of_week,is_weekend,is_rush_hour
0,NaN,14.06,0.0,0.0,45,Clouds,few clouds,2022-01-01,Cairo-Suez Road,Gateway,903,2022,1,1,0,5,1,0
1,NaN,16.38,0.0,0.0,58,Clouds,few clouds,2022-01-01,Green River Corridor,Internal,258,2022,1,1,0,5,1,0
2,NaN,13.42,0.0,0.0,9,Clear,clear sky,2022-01-01,Regional Ring Road,Gateway,1016,2022,1,1,0,5,1,0
3,NaN,11.32,0.0,0.0,14,Clear,clear sky,2022-01-01,Mohamed Bin Zayed Axis (North),Gateway,555,2022,1,1,0,5,1,0
4,NaN,13.13,0.0,0.0,33,Clouds,broken clouds,2022-01-01,Cairo-Ain Sokhna Road,Gateway,551,2022,1,1,0,5,1,0


## 6. Define Features, Target, and Train/Validation/Test Split
The split is time-based (80% / 10% / 10%) to avoid leaking future traffic patterns into the training set.

In [11]:
TARGET = "traffic_volume"

features = [
    "temp", "rain_1h", "clouds_all", "holiday", "weather_main",
    "weather_description", "road_name", "year", "month", "day",
    "hour", "day_of_week", "is_weekend", "is_rush_hour",
]

numeric_features = [
    "temp", "rain_1h", "clouds_all", "year", "month", "day",
    "hour", "day_of_week", "is_weekend", "is_rush_hour",
]

categorical_features = [
    "holiday", "weather_main", "weather_description", "road_name",
]

X = df[features].copy()
y = df[TARGET].copy()

mask = y.notna()
X = X.loc[mask].reset_index(drop=True)
y = y.loc[mask].reset_index(drop=True)

train_end = int(len(X) * 0.80)
val_end = int(len(X) * 0.90)

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (122046, 14)
Validation: (15256, 14)
Testing: (15256, 14)


## 7. Preprocessing Pipeline

In [12]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)

Processed training shape: (122046, 47)


## 8. Evaluation Helper

In [13]:
def evaluate_model(model_name, y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{model_name}")
    print("-" * 40)
    print(f"MAE  : {mae:.2f}")
    print(f"RMSE : {rmse:.2f}")
    print(f"R2   : {r2:.4f}")

    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2}


results = []

## 9. Linear Regression (Baseline)

In [14]:
linear_model = LinearRegression()
linear_model.fit(X_train_processed, y_train)
y_pred_linear = linear_model.predict(X_test_processed)
results.append(evaluate_model("Linear Regression", y_test, y_pred_linear))


Linear Regression
----------------------------------------
MAE  : 331.99
RMSE : 409.44
R2   : 0.6257


## 10. Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=120,
    max_depth=18,
    min_samples_split=5,
    random_state=42,
    n_jobs=1,
)
rf_model.fit(X_train_processed, y_train)
y_pred_rf = rf_model.predict(X_test_processed)
results.append(evaluate_model("Random Forest", y_test, y_pred_rf))

## 11. Gradient Boosting

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
)
gb_model.fit(X_train_processed, y_train)
y_pred_gb = gb_model.predict(X_test_processed)
results.append(evaluate_model("Gradient Boosting", y_test, y_pred_gb))


Gradient Boosting
----------------------------------------
MAE  : 139.67
RMSE : 194.98
R2   : 0.9151


## 12. Artificial Neural Network (ANN)

In [ ]:
X_train_ann = np.asarray(X_train_processed, dtype=np.float32)
X_val_ann = np.asarray(X_val_processed, dtype=np.float32)
X_test_ann = np.asarray(X_test_processed, dtype=np.float32)

y_train_ann = np.asarray(y_train, dtype=np.float32)
y_val_ann = np.asarray(y_val, dtype=np.float32)
y_test_ann = np.asarray(y_test, dtype=np.float32)

ann_model = keras.Sequential([
    keras.Input(shape=(X_train_ann.shape[1],)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),
])

ann_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

ann_model.fit(
    X_train_ann, y_train_ann,
    validation_data=(X_val_ann, y_val_ann),
    epochs=25,
    batch_size=512,
    callbacks=[early_stopping],
    verbose=2,
)

Epoch 1/25
239/239 - 2s - 10ms/step - loss: 645162.5000 - mae: 597.5728 - val_loss: 238047.0312 - val_mae: 404.0593
Epoch 2/25
239/239 - 1s - 4ms/step - loss: 192040.6406 - mae: 351.8901 - val_loss: 166212.2188 - val_mae: 333.0946
Epoch 3/25
239/239 - 1s - 4ms/step - loss: 157748.7344 - mae: 314.2924 - val_loss: 143347.0156 - val_mae: 307.1401
Epoch 4/25
239/239 - 1s - 4ms/step - loss: 127336.2500 - mae: 277.6723 - val_loss: 91919.6719 - val_mae: 232.8212
Epoch 5/25
239/239 - 1s - 4ms/step - loss: 82647.5703 - mae: 214.8845 - val_loss: 53258.1797 - val_mae: 168.0188
Epoch 6/25
239/239 - 1s - 4ms/step - loss: 64576.3320 - mae: 184.4509 - val_loss: 44827.9336 - val_mae: 149.7023
Epoch 7/25
239/239 - 1s - 4ms/step - loss: 57765.0469 - mae: 173.2545 - val_loss: 39985.4141 - val_mae: 143.1059
Epoch 8/25
239/239 - 1s - 4ms/step - loss: 52948.6953 - mae: 165.5783 - val_loss: 36765.2930 - val_mae: 136.6036
Epoch 9/25
239/239 - 1s - 4ms/step - loss: 49317.7539 - mae: 159.0984 - val_loss: 32330.

In [ ]:
y_pred_ann = ann_model.predict(X_test_ann, verbose=0).flatten()
results.append(evaluate_model("ANN", y_test_ann, y_pred_ann))


ANN
----------------------------------------
MAE  : 98.43
RMSE : 135.17
R2   : 0.9592


## 13. LSTM (Sequence Model)
Trained here for comparison in the model performance chart. It is not used for live predictions in `app.py`, only the Random Forest, Gradient Boosting, and ANN models are.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

lstm_features = [
    "traffic_volume", "temp", "rain_1h", "clouds_all",
    "hour", "day_of_week", "is_weekend", "is_rush_hour",
]

lstm_df = df[["date_time"] + lstm_features].copy()
lstm_df = lstm_df.sort_values("date_time").reset_index(drop=True)
lstm_df = lstm_df.dropna().reset_index(drop=True)

train_end_lstm = int(len(lstm_df) * 0.80)
val_end_lstm = int(len(lstm_df) * 0.90)

lstm_train = lstm_df.iloc[:train_end_lstm].copy()
lstm_val = lstm_df.iloc[train_end_lstm:val_end_lstm].copy()
lstm_test = lstm_df.iloc[val_end_lstm:].copy()

lstm_scaler = MinMaxScaler()

lstm_train_scaled = lstm_scaler.fit_transform(lstm_train[lstm_features])
lstm_val_scaled = lstm_scaler.transform(lstm_val[lstm_features])
lstm_test_scaled = lstm_scaler.transform(lstm_test[lstm_features])

WINDOW_SIZE = 24
TARGET_INDEX = 0


def create_sequences(data, window_size):
    X_seq, y_seq = [], []
    for i in range(window_size, len(data)):
        X_seq.append(data[i - window_size:i])
        y_seq.append(data[i, TARGET_INDEX])
    return np.array(X_seq), np.array(y_seq)


X_lstm_train, y_lstm_train = create_sequences(lstm_train_scaled, WINDOW_SIZE)
X_lstm_val, y_lstm_val = create_sequences(lstm_val_scaled, WINDOW_SIZE)
X_lstm_test, y_lstm_test = create_sequences(lstm_test_scaled, WINDOW_SIZE)

lstm_model = keras.Sequential([
    keras.Input(shape=(X_lstm_train.shape[1], X_lstm_train.shape[2])),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation="relu"),
    layers.Dense(1),
])

lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
)

early_stopping_lstm = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

lstm_model.fit(
    X_lstm_train, y_lstm_train,
    validation_data=(X_lstm_val, y_lstm_val),
    epochs=20,
    batch_size=512,
    callbacks=[early_stopping_lstm],
    verbose=2,
)

Epoch 1/20
239/239 - 28s - 118ms/step - loss: 0.0137 - mae: 0.0870 - val_loss: 0.0125 - val_mae: 0.0821
Epoch 2/20
239/239 - 25s - 105ms/step - loss: 0.0123 - mae: 0.0820 - val_loss: 0.0128 - val_mae: 0.0811
Epoch 3/20
239/239 - 22s - 91ms/step - loss: 0.0121 - mae: 0.0813 - val_loss: 0.0125 - val_mae: 0.0807
Epoch 4/20
239/239 - 26s - 107ms/step - loss: 0.0119 - mae: 0.0806 - val_loss: 0.0126 - val_mae: 0.0803
Epoch 5/20
239/239 - 25s - 106ms/step - loss: 0.0119 - mae: 0.0804 - val_loss: 0.0121 - val_mae: 0.0805
Epoch 6/20
239/239 - 26s - 110ms/step - loss: 0.0118 - mae: 0.0799 - val_loss: 0.0121 - val_mae: 0.0795
Epoch 7/20
239/239 - 24s - 102ms/step - loss: 0.0117 - mae: 0.0796 - val_loss: 0.0119 - val_mae: 0.0801
Epoch 8/20
239/239 - 21s - 87ms/step - loss: 0.0116 - mae: 0.0793 - val_loss: 0.0123 - val_mae: 0.0793
Epoch 9/20
239/239 - 22s - 91ms/step - loss: 0.0116 - mae: 0.0792 - val_loss: 0.0119 - val_mae: 0.0793
Epoch 10/20
239/239 - 20s - 85ms/step - loss: 0.0115 - mae: 0.0789 

In [ ]:
def inverse_target_scaling(scaled_values, scaler, n_features):
    dummy = np.zeros((len(scaled_values), n_features))
    dummy[:, 0] = scaled_values
    original = scaler.inverse_transform(dummy)
    return original[:, 0]


y_pred_lstm_scaled = lstm_model.predict(X_lstm_test, verbose=0).flatten()

y_pred_lstm = inverse_target_scaling(y_pred_lstm_scaled, lstm_scaler, len(lstm_features))
y_true_lstm = inverse_target_scaling(y_lstm_test, lstm_scaler, len(lstm_features))

results.append(evaluate_model("LSTM", y_true_lstm, y_pred_lstm))


LSTM
----------------------------------------
MAE  : 334.77
RMSE : 453.66
R2   : 0.5407


## 14. Final Model Comparison

In [ ]:
final_results = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
final_results

,Model,MAE,RMSE,R2
0,Random Forest,66.593662,102.832369,0.976390
1,ANN,98.426346,135.166448,0.959208
2,Gradient Boosting,139.670710,194.984702,0.915113
3,Linear Regression,331.986172,409.435652,0.625707
4,LSTM,334.773392,453.656952,0.540737


## 15. Save Artifacts Used by `app.py`
The ANN is saved as **weights-only** (`ann_model.weights.h5`) rather than a full `.keras` file. `app.py` rebuilds the same architecture in code and loads these weights — this avoids Keras full-model deserialization errors that can occur when the app runs on a different Keras/TensorFlow version than the one used for training.

The LSTM model and its scaler are **not saved** here, since `app.py` only uses Random Forest, Gradient Boosting, and ANN for live predictions. The LSTM's evaluation metrics above are already included in `model_results.csv` for the model comparison chart.

In [ ]:
joblib.dump(rf_model, "rf_model.pkl")
joblib.dump(gb_model, "gb_model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

ann_model.save_weights("ann_model.weights.h5")

final_results.to_csv("model_results.csv", index=False)

print("All artifacts used by app.py saved successfully!")

All artifacts used by app.py saved successfully!
